# Daily Challenge – Week 3, Day 3: Analysis of Airplane Crashes and Fatalities (1908–2023)
**Libraries used:** Pandas (loading, cleaning, grouping), NumPy (calculations), SciPy (statistics and hypothesis tests), Matplotlib & Seaborn (charts).

Structure: 1) import & cleaning · 2) exploratory analysis · 3) statistical analysis · 4) visualization · 5) insights report.

## 1. Data Import and Cleaning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# The file is not UTF-8 (it contains characters like "ñ" in Latin-1), so we tell pandas the encoding
df = pd.read_csv("Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv", encoding="latin-1")
print(df.shape)
df.head(3)

In [ ]:
# Missing values per column
df.isnull().sum()

**Cleaning decisions:**
- `Date` → real datetime; then extract `Year` and `Decade`.
- `Aboard` / `Fatalities` are the key numbers. Only 18 and 8 rows are missing → **drop those rows** (too few to matter, and guessing them would distort totals).
- `Ground` (people killed on the ground) – missing means none were reported → fill with 0.
- Columns with many gaps that we don't analyze (`Time`, `Flight #`, `Route`, `cn/ln`…) are left as they are.
- New categories: **`Country`** (last part of `Location`; US states grouped as "USA"), **`Type`** (Military vs. Civil), and **`SurvivalRate`** per crash.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
df["Year"] = df["Date"].dt.year
df["Decade"] = (df["Year"] // 10) * 10

df = df.dropna(subset=["Aboard", "Fatalities"])
df["Ground"] = df["Ground"].fillna(0)

# Country / region from the location
us_states = ["Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut","Delaware","Florida",
             "Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa","Kansas","Kentucky","Louisiana","Maine",
             "Maryland","Massachusetts","Michigan","Minnesota","Mississippi","Missouri","Montana","Nebraska",
             "Nevada","New Hampshire","New Jersey","New Mexico","New York","North Carolina","North Dakota","Ohio",
             "Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina","South Dakota","Tennessee","Texas",
             "Utah","Vermont","Virginia","Washington","West Virginia","Wisconsin","Wyoming","D.C.","Washington D.C."]
df["Country"] = df["Location"].str.split(",").str[-1].str.strip()
df.loc[df["Country"].isin(us_states), "Country"] = "USA"

df["Type"] = np.where(df["Operator"].str.contains("Military", na=False), "Military", "Civil")

# Survival rate per crash (only where someone was aboard)
df = df[df["Aboard"] > 0]
df["Survivors"] = (df["Aboard"] - df["Fatalities"]).clip(lower=0)
df["SurvivalRate"] = df["Survivors"] / df["Aboard"]

print("Rows after cleaning:", len(df))
df[["Date", "Country", "Type", "Aboard", "Fatalities", "SurvivalRate"]].head()

## 2. Exploratory Data Analysis

In [ ]:
total_crashes = len(df)
total_aboard = df["Aboard"].sum()
total_fatalities = df["Fatalities"].sum()

print("Number of crashes:", total_crashes)
print("People aboard:", int(total_aboard))
print("Fatalities (aboard):", int(total_fatalities))
print("Overall survival rate:", round(1 - total_fatalities / total_aboard, 3))
print("Crashes with no survivors:", round((df["Survivors"] == 0).mean() * 100, 1), "%")

In [ ]:
by_decade = df.groupby("Decade").agg(
    crashes=("Date", "size"),
    fatalities=("Fatalities", "sum"),
    avg_fatalities=("Fatalities", "mean"),
    aboard=("Aboard", "sum"),
)
by_decade["survival_rate"] = 1 - by_decade["fatalities"] / by_decade["aboard"]
by_decade.round(2)

In [ ]:
df["Country"].value_counts().head(10)

## 3. Statistical Analysis (SciPy)

In [ ]:
fatalities = df["Fatalities"]
print("Fatalities per crash – mean:", round(fatalities.mean(), 2),
      "| median:", fatalities.median(),
      "| std:", round(fatalities.std(), 2))
print("Skewness:", round(stats.skew(fatalities), 2), "(> 0 means a long tail of very deadly crashes)")

survival = df["SurvivalRate"]
print("\nSurvival rate per crash – mean:", round(survival.mean(), 3),
      "| median:", survival.median(),
      "| std:", round(survival.std(), 3))

### Hypothesis test 1 – Are crashes in the 2000s less deadly than in the 1970s?
- **H0:** the average number of fatalities per crash is the same in both decades.
- **H1:** it is different.

Because fatalities are very skewed, I run both a **Welch t-test** (compares means) and a **Mann-Whitney U test** (doesn't assume a normal distribution).

In [ ]:
f_1970s = df.loc[df["Decade"] == 1970, "Fatalities"]
f_2000s = df.loc[df["Decade"] == 2000, "Fatalities"]

print("Mean 1970s:", round(f_1970s.mean(), 2), "| Mean 2000s:", round(f_2000s.mean(), 2))

t_stat, p_t = stats.ttest_ind(f_1970s, f_2000s, equal_var=False)
u_stat, p_u = stats.mannwhitneyu(f_1970s, f_2000s)
print("Welch t-test p-value:", f"{p_t:.2e}")
print("Mann-Whitney p-value:", f"{p_u:.2e}")

### Hypothesis test 2 – Do military and civil crashes differ in survival rate?

In [ ]:
surv_mil = df.loc[df["Type"] == "Military", "SurvivalRate"]
surv_civ = df.loc[df["Type"] == "Civil", "SurvivalRate"]

print("Mean survival – Military:", round(surv_mil.mean(), 3), "| Civil:", round(surv_civ.mean(), 3))
u_stat, p_u2 = stats.mannwhitneyu(surv_mil, surv_civ)
print("Mann-Whitney p-value:", p_u2)

## 4. Visualization

In [ ]:
crashes_per_year = df.groupby("Year").size()

plt.figure(figsize=(12, 4))
plt.plot(crashes_per_year.index, crashes_per_year.values)
plt.title("Number of crashes per year (1908–2023)")
plt.xlabel("Year")
plt.ylabel("Crashes")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
by_decade["avg_fatalities"].plot(kind="bar", ax=axes[0])
axes[0].set_title("Average fatalities per crash, by decade")
by_decade["survival_rate"].plot(kind="bar", ax=axes[1], color="seagreen")
axes[1].set_title("Survival rate, by decade")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
top_countries = df["Country"].value_counts().head(10)
sns.barplot(x=top_countries.values, y=top_countries.index)
plt.title("Top 10 countries by number of crashes")
plt.xlabel("Crashes")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["Fatalities"], bins=60, ax=axes[0])
axes[0].set_title("Fatalities per crash (very skewed)")
sns.histplot(df["SurvivalRate"], bins=20, ax=axes[1])
axes[1].set_title("Survival rate per crash")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="Type", y="SurvivalRate")
plt.title("Survival rate: military vs. civil crashes")
plt.show()

## 5. Insights and Report
**How the tools were used:** Pandas for loading (with the right file encoding), cleaning, new categories and group-by summaries; NumPy for vectorized calculations (military/civil labels, survival counts); SciPy for descriptive statistics, skewness and two hypothesis tests; Matplotlib and Seaborn for the charts.

**Key findings**
1. **Scale:** about **4,975 crashes** with **155,000 people aboard**, of whom **111,500 died**. Overall, **28%** of people aboard survived — and in **64% of crashes nobody survived**. (The dataset focuses on serious, fatal accidents, so it says nothing about how safe flying is overall.)
2. **Crashes over time:** the number of crashes climbed until the 1940s, stayed high (about 550–650 per decade) from the 1940s to the 1990s, then **fell sharply**: 506 in the 2000s and 235 in the 2010s. The 2020s only include data up to late 2023.
3. **Deadlier crashes in the 1970s, fewer deaths per crash since:** the average number of deaths per crash peaked in the **1970s (~32)**, when large jets became common, and dropped to **~22 in the 2000s**. Both the Welch t-test and the Mann-Whitney test give p-values well below 0.05, so this difference is **statistically significant**.
4. **Survival improved:** the share of people aboard who survived rose from roughly **19–23% before 1970** to **32–36% in the 1980s–2000s**.
5. **Very skewed distribution:** median 11 deaths per crash vs. mean 22.4, skewness 4.6. A few catastrophic crashes (e.g. Tenerife 1977, 583 deaths) pull the average up, so the **median describes a "typical" crash better**.
6. **Where:** the **USA** has by far the most crashes (about 1,000), followed by Russia (257) and Brazil (175). Without data on the number of flights per country, this shows where the most flying happens, **not** where flying is most dangerous.
7. **Military vs. civil:** military flights are 15% of the crashes, and their average survival rate is much lower (**6.5% vs. 20%**, p far below 0.05).

**Anomalies / data-quality notes**
- The `Ground` column has **2,750 ground deaths listed twice** — on both planes of 11 September 2001 — so summing that column would double-count them.
- Column names contain typos ("Passangers"), the file isn't UTF-8, and location formats vary (US states vs. countries), which needed cleaning before grouping.
- Small numbers of rows with missing `Aboard`/`Fatalities` were removed.

**Conclusion:** aviation crashes became rarer after the 1990s, and when they happen, a larger share of people survive — consistent with improvements in aircraft, regulation and emergency response. A fair comparison of risk between decades or countries would need the number of flights, which this dataset doesn't include.
